# GReLU neuron on XX vs XXX thermal states: Algorithm 8 only

This notebook trains one quantized GReLU neuron with **Algorithm 8** (squared loss) to tell thermal states of random 10-qubit XX chains from those of XXX chains. Then it tests the trained neuron. Every function is defined in the notebook; nothing is hidden in an `include`.

1. **The neuron**: GReLU, the Hamiltonian H(θ), the exact loss and gradient, and firing (Algorithm 5).
2. **The Algorithm 8 circuit, simulated**: one run of the circuit, shot by shot, and the three places where θ enters it.
3. **The data**: 840 thermal states, rebuilt exactly, then split at random into train and test states.
4. **Training**: Adam with the exact gradient. θ and its gradient are recorded at every step.
5. **Circuit check**: at a few training steps, the simulated circuit's gradient is compared with the exact gradient that training used.
6. **Testing**: exact test accuracy, then classification by firing the neuron with finite copies of each state.
7. **Figures**, then results.

Paper: He, Liu & Wilde, *Fermi–Dirac machines as quantizations of neurons* (arXiv:2605.24386), Sec. IV.B and App. F. The margin-loss route (Algorithm 9) is left out here; the earlier two-algorithm version is in `archive/`, and folder 2 has the full study.

**Running.** Use the "Julia 1.12" kernel, from anywhere inside the repository. The whole notebook takes about 10 minutes. It needs HDF5, Optimisers, Plots and SpecialFunctions. CSVs go to `results/` and figures to `figures/`.

**The same code as scripts.** `neuron.jl`, `data.jl`, `run.jl` and `plots.jl` hold exactly the code cells below (`julia 3_xx_xxx_grelu_minimal/run.jl` runs everything from the repository root).

In [ ]:
# Paths: find the repository root (the folder that contains data/xx_xxx_thermal_states)
function find_root(dir=pwd())
  while !isdir(joinpath(dir, "data", "xx_xxx_thermal_states"))
    parent = dirname(dir)
    parent == dir && error("start the notebook from inside the repository")
    dir = parent
  end
  return dir
end
const ROOT = find_root()
const HERE = joinpath(ROOT, "3_xx_xxx_grelu_minimal")
println("repository: ", ROOT)

## 1. The neuron

The header comment below sums up the model: what the neuron computes, how Algorithm 8 trains it, and how firing works.

### Activation
GReLU_T(x) = x Φ(x/T) + T φ(x/T) is a ReLU smoothed by a Gaussian of width T.

In [ ]:
# neuron.jl — the quantized GReLU neuron (He, Liu & Wilde, "Fermi–Dirac machines as
# quantizations of neurons", arXiv:2605.24386, Sec. IV.B), trained with
# Algorithm 8, and the Algorithm 8 circuit simulated shot by shot.
#
# A classical neuron computes ReLU(w·x).  The quantum version takes a density
# matrix rho and a Hamiltonian built from trainable weights,
#
#     H(theta) = theta_0 I + sum_j theta_j H_j     (H_j = Pauli strings: Z, ZZ, XX, YY)
#
# and its output is   Tr[ GReLU_T(H(theta)) rho ],   where
#
#     GReLU_T(x) = x Phi(x/T) + T phi(x/T)      (a smoothed ReLU; Phi, phi =
#                                               standard normal cdf and pdf).
#
# Training (Algorithm 8).  Squared loss
#
#     L(theta) = mean_m (Tr[GReLU_T(H) rho_m] - t_m)^2,    t = 0 (XX) or 1 (XXX).
#
# Algorithm 8 is a quantum circuit whose measured outcomes average to dL/dtheta
# (Section 2 below simulates it shot by shot).  Training uses that average
# directly: the EXACT gradient, which is what the circuit gives in the limit
# of infinitely many runs.  The identity term theta_0 is a bias, needed so the
# output can sit near 0 for XX states and near 1 for XXX states.
#
# Classification.  Predict XXX when the output is > 1/2.  Algorithm 5
# ("firing" the neuron) uses ONE copy of rho per run and returns a random
# number whose average is the neuron output.  In the eigenbasis of H, one run
# is: pick an eigenvalue E with probability <E|rho|E>, then return
# ReLU(E + T z) with z ~ N(0, 1).

using LinearAlgebra, Random, SparseArrays, Statistics
using SpecialFunctions: erf

# ------------------------------------------------------------- activation ---

Phi(x) = (1 + erf(x / sqrt(2))) / 2
phi(x) = exp(-x^2 / 2) / sqrt(2pi)
grelu(x, T) = x * Phi(x / T) + T * phi(x / T)

### Pauli strings and H(θ)
Each term Hⱼ is a sparse 1024 × 1024 Pauli string. The model has 38 terms: the identity (a bias), 10 Z, 9 ZZ, 9 XX and 9 YY.

In [ ]:
# ---------------------------------------------------- Pauli strings and H ---

const PAULI = Dict('I' => ComplexF64[1 0; 0 1], 'X' => ComplexF64[0 1; 1 0],
                   'Y' => ComplexF64[0 -im; im 0], 'Z' => ComplexF64[1 0; 0 -1])

"Sparse matrix of a Pauli string on n qubits, e.g. ops = [(3, 'X'), (4, 'X')]."
function pauli(n, ops)
  factors = [sparse(PAULI['I']) for _ in 1:n]
  for (site, P) in ops
    factors[site] = sparse(PAULI[P])
  end
  return real(reduce(kron, factors))        # every string we use is real
end

"""
The neuron's terms: the identity (bias), then Z_i, Z_iZ_{i+1}, X_iX_{i+1},
Y_iY_{i+1} (38 terms at n = 10).  Returns (names, matrices).
"""
function neuron_terms(n)
  names, ops = String["I"], Vector{Tuple{Int,Char}}[Tuple{Int,Char}[]]
  for i in 1:n
    push!(names, "Z$i"); push!(ops, [(i, 'Z')])
  end
  for P in "ZXY", i in 1:(n - 1)
    push!(names, "$P$i$P$(i+1)"); push!(ops, [(i, P), (i + 1, P)])
  end
  return names, [pauli(n, o) for o in ops]
end

hamiltonian(terms, theta) = Matrix(sum(t .* Hj for (t, Hj) in zip(theta, terms)))

### The states, seen by H
Each state is ρ = W diag(p) Wᵀ, using its chain's eigenvectors W. To evaluate the neuron we only need the state's populations on the eigenvectors of H(θ).

In [ ]:
# ------------------------------------------------- the states, seen by H ---
#
# Every input state is rho_m = W diag(p_m) W' (its chain's eigenvectors W and
# Boltzmann weights p_m; see the data section).  With H = V diag(E) V', the
# populations of rho_m on H's eigenvectors are (A.^2) p_m, with A = V'W.

"For each state m in `idx`: its populations <E|rho_m|E> on the eigenvectors V of H."
function populations(ds, idx, V)
  pops = zeros(size(V, 1), length(idx))
  for c in unique(ds.chain[idx])
    A2 = (V' * ds.chains[c].W) .^ 2
    for (i, m) in enumerate(idx)
      ds.chain[m] == c && (pops[:, i] = A2 * boltzmann(ds.chains[c], ds.beta[m]))
    end
  end
  return pops
end

targets(ds, idx) = (ds.y[idx] .+ 1) ./ 2          # 0 for XX, 1 for XXX

### Exact loss and gradient
The Daleckii–Krein formula differentiates Tr[f(H) ρ] exactly. The result is the average of what the Algorithm 8 circuit measures, in the limit of infinitely many runs.

In [ ]:
# ------------------------------------------------ exact loss and gradient ---
#
# Derivative of a matrix function (Daleckii–Krein):
#   d/dtheta_j Tr[f(H) R] = Tr[H_j V (G ∘ V'RV) V'],   G_ab = (f(E_a) - f(E_b)) / (E_a - E_b)

function divided_differences(E, T)
  G = similar(E, length(E), length(E))
  for (k, b) in enumerate(E), (i, a) in enumerate(E)
    G[i, k] = abs(a - b) < 1e-9 ? Phi((a + b) / 2T) :          # a ≈ b: f'(a) = Phi(a/T)
                                  (grelu(a, T) - grelu(b, T)) / (a - b)
  end
  return G
end

gradient_from(terms, V, K) = (M = V * K * V'; [dot(Hj, M) for Hj in terms])


"Algorithm 8's loss and its exact gradient, on the states `idx`."
function square_loss_grad(ds, idx, terms, theta, T)
  E, V = eigen(Symmetric(hamiltonian(terms, theta)))
  chains = unique(ds.chain[idx])
  A = Dict(c => V' * ds.chains[c].W for c in chains)     # each chain's eigenvectors, seen by H
  p = [boltzmann(ds.chains[ds.chain[m]], ds.beta[m]) for m in idx]
  out = [dot(grelu.(E, T), (A[ds.chain[m]] .^ 2) * p[i]) for (i, m) in enumerate(idx)]
  r = out .- targets(ds, idx)                            # residuals vs targets {0, 1}
  # gradient of mean(r^2) = gradient of Tr[GReLU(H) Rc] with Rc = (2/M) sum_m r_m rho_m
  Rc = zeros(size(V))
  for c in chains
    q = sum(2r[i] / length(idx) .* p[i] for (i, m) in enumerate(idx) if ds.chain[m] == c)
    Rc .+= A[c] * Diagonal(q) * A[c]'
  end
  return mean(abs2, r), gradient_from(terms, V, divided_differences(E, T) .* Rc)
end

"""
Exact decision scores on the states `idx` (the infinite-copy limit):
Tr[GReLU(H) rho] - 1/2.  Predict XXX (+1) when the score is >= 0.
"""
function scores(ds, idx, terms, theta, T)
  E, V = eigen(Symmetric(hamiltonian(terms, theta)))
  return populations(ds, idx, V)' * grelu.(E, T) .- 0.5
end

### Firing (Algorithm 5)
One firing uses one copy of ρ. It picks an eigenvalue E of H with probability ⟨E|ρ|E⟩, then outputs ReLU(E + T z) with z ~ N(0, 1). Its mean is the neuron output.

In [ ]:
# ---------------------------------------------------- firing (Algorithm 5) ---

"One firing of the neuron on one copy of the state: ReLU(E + T z)."
function fire(E, cdf, T, rng)
  i = min(searchsortedfirst(cdf, rand(rng)), length(E))
  return max(E[i] + T * randn(rng), 0.0)
end

"Classify one state from `ncopies` firings: XXX (+1) if the average output is > 1/2."
function classify_by_firing(E, pop, T, ncopies, rng)
  cdf = cumsum(max.(pop, 0) ./ sum(max.(pop, 0)))
  return mean(fire(E, cdf, T, rng) for _ in 1:ncopies) > 0.5 ? 1.0 : -1.0
end

## 2. The Algorithm 8 circuit, simulated

Training below uses the exact gradient. This section simulates the circuit that the exact gradient stands in for. Each run of Algorithm 8 picks a training state m at random and uses **two independent copies** of ρ_m:

- **Value block** (copy 1) gives b₁, an unbiased estimate of Tr[GReLU(H) ρ_m] − t_m. It measures one Pauli term H_k and runs one Hadamard test with the evolution e^{−iλvtH}. This rests on Tr[GReLU(H)ρ] = T/√(2π) + ½Tr[Hρ] + (‖θ‖₁²/√(2π)T) E[λ Re Tr[H² e^{−iλvtH} ρ]/‖θ‖₁²] (paper Lemma 5).
- **Gradient block** (copy 2) gives b₂ⱼ for every j, an unbiased estimate of ∂ⱼ Tr[GReLU(H) ρ_m]. It measures H_j and runs the Theorem 17 Hadamard test (paper Fig. 11), with controlled H_j e^{iHτ} on a state evolved by e^{−iHsτ}.
- The run outputs **2 b₁ b₂ⱼ**. Since the two copies are independent, its mean is 2 (Tr[GReLU(H)ρ_m] − t_m) ∂ⱼTr[GReLU(H)ρ_m], which averaged over m is ∂ⱼL.

**Where θ enters the circuit.** The circuit is rebuilt from the current θ at every Adam step, in three ways:
1. **H(θ) sets every time evolution** (e^{−iλvtH}, e^{iHτ}, e^{−iHsτ}).
2. **θ sets which term is measured**: the term index k is drawn with probability q(k) = |θ_k| / ‖θ‖₁, and the outcome is multiplied by sgn(θ_k).
3. **‖θ‖₁ scales the outputs**: by ‖θ‖₁/2 and ‖θ‖₁²/(√(2π)T) in the value block and by √(2/π)‖θ‖₁/T in the gradient block. A larger ‖θ‖₁ means noisier single runs.

**How it is simulated.** Each Hadamard test is evaluated exactly in the eigenbasis of H(θ), where every evolution is a diagonal phase. Then a single ±1 outcome is drawn with the right mean. This gives exactly the outcome distribution of one run of the real circuit. The value block's Hadamard test is diagonal in H's eigenbasis, so it needs only the populations ⟨E|ρ_m|E⟩. The code is ported from `2_xx_xxx_grelu_study/src/grelu_neuron.jl` (`alg8_grelu_gradient`) to the plain dense matrices used here.

In [ ]:
# -------------------------------------------- Algorithm 8, shot by shot ---

"One run of a circuit whose ±1 outcome has mean e."
shot(rng, e) = rand(rng) < (1 + clamp(e, -1, 1)) / 2 ? 1.0 : -1.0

"Draw a term index k with probability |theta_k| / ||theta||_1 (cq = its cumulative sum)."
sampleq(rng, cq) = min(searchsortedfirst(cq, rand(rng)), length(cq))

"Tr[H_j rho_m] for every state m in `idx` and term j (does not depend on theta)."
function pauli_expectations(ds, idx, terms)
  C = zeros(length(idx), length(terms))
  for c in unique(ds.chain[idx])
    W = ds.chains[c].W
    D = [vec(sum(W .* (Hj * W); dims=1)) for Hj in terms]   # <w|H_j|w> per eigenvector
    for (i, m) in enumerate(idx)
      ds.chain[m] == c || continue
      p = boltzmann(ds.chains[c], ds.beta[m])
      C[i, :] = [dot(p, d) for d in D]
    end
  end
  return C
end

"""
Value block, one run on one copy of rho_m: an unbiased estimate of Tr[GReLU_T(H) rho_m].
`E2pop` = E.^2 .* <E|rho_m|E>,  `Cm` = Tr[H_j rho_m].
"""
function value_shot(E, E2pop, Cm, theta, cq, T, rng)
  nt1 = norm(theta, 1)
  k = sampleq(rng, cq)                                     # theta picks the measured term
  a = nt1 / 2 * sign(theta[k]) * shot(rng, Cm[k])           # estimates Tr[H rho] / 2
  lam, v, t = rand(rng), rand(rng), randn(rng) / T
  e = sum(E2pop[i] * cos(lam * v * t * E[i]) for i in eachindex(E)) / nt1^2   # Hadamard test
  return T / sqrt(2pi) + a + nt1^2 / (sqrt(2pi) * T) * lam * shot(rng, e)
end

"""
The value block alone, run many times on one state m: the running estimate of
Tr[GReLU_T(H) rho_m] after each number of runs in `runs`, and the exact value.
"""
function value_block_estimates(ds, m, Cm, terms, theta, T; runs, rng)
  E, V = eigen(Symmetric(hamiltonian(terms, theta)))
  pop = populations(ds, [m], V)[:, 1]
  E2pop, cq = E .^ 2 .* pop, cumsum(abs.(theta) ./ norm(theta, 1))
  total, out = 0.0, Float64[]
  for n in 1:maximum(runs)
    total += value_shot(E, E2pop, Cm, theta, cq, T, rng)
    n in runs && push!(out, total / n)
  end
  return out, dot(grelu.(E, T), pop)
end

"""
Gradient block's Hadamard test (Theorem 17, paper Fig. 11), for every j at once:
Re Tr[H_k H_j e^{iH tau} e^{-iH s tau} rho e^{iH s tau}].  `Rt` = V' rho V.
"""
function hadamard_thm17(terms, E, V, Rt, s, tau, k)
  X = V * (Rt .* cos.(tau .* ((1 - s) .* E .+ s .* E'))) * V'
  Y = X * terms[k]
  return [dot(Hj, Y) for Hj in terms]                      # Tr[H_j X H_k]
end

"""
Simulated Algorithm 8: `nruns` runs of the circuit on the training states `idx`.
Each run draws a state m and outputs 2 b1 b2_j for every j, from two independent
copies of rho_m.  Returns the outputs as a (terms × runs) matrix; their mean
estimates the squared-loss gradient.
"""
function alg8_circuit_runs(ds, idx, C, terms, theta, T; nruns, rng)
  E, V = eigen(Symmetric(hamiltonian(terms, theta)))         # the circuit is built from H(theta)
  A = Dict(c => V' * ds.chains[c].W for c in unique(ds.chain[idx]))
  t = targets(ds, idx)
  nt1 = norm(theta, 1)
  cq = cumsum(abs.(theta) ./ nt1)
  pref = sqrt(2 / pi) * nt1 / T
  out = zeros(length(theta), nruns)
  for r in 1:nruns
    i = rand(rng, eachindex(idx))
    Ac, p = A[ds.chain[idx[i]]], boltzmann(ds.chains[ds.chain[idx[i]]], ds.beta[idx[i]])
    # copy 1: value block
    b1 = value_shot(E, E .^ 2 .* ((Ac .^ 2) * p), C[i, :], theta, cq, T, rng) - t[i]
    # copy 2: gradient block
    s, v, tt = rand(rng), rand(rng), randn(rng) / T
    k = sampleq(rng, cq)
    e = hadamard_thm17(terms, E, V, Ac * Diagonal(p) * Ac', s, v * tt, k)
    for j in eachindex(theta)
      b2 = shot(rng, C[i, j]) / 2 + pref * s * sign(theta[k]) * shot(rng, e[j])
      out[j, r] = 2 * b1 * b2
    end
  end
  return out
end

## 3. The data

The dataset holds 84 random chains, 42 XX and 42 XXX, each at 10 temperatures.

In [ ]:
# data.jl — load the XX vs XXX thermal states and split them into train / test.
#
# The dataset (data/xx_xxx_thermal_states/xx_xxx_n10.h5) holds 840 thermal
# states of random 10-qubit spin chains:
#
#   XX  chain:  H = sum_i J_i (X_i X_{i+1} + Y_i Y_{i+1})                label y = -1
#   XXX chain:  H = sum_i J_i (X_i X_{i+1} + Y_i Y_{i+1} + Z_i Z_{i+1})  label y = +1
#
# 84 chains (42 of each kind), each at 10 temperatures kT.  The file also ships
# each state as an MPO, but we rebuild every state exactly from its couplings
# J instead: rho = exp(-H/kT) / Z.  We diagonalise each chain once,
# H = W diag(eps) W', and then every temperature is just a different set of
# Boltzmann weights on the same eigenvectors.

using HDF5, LinearAlgebra, Random

### One chain = its eigenvectors
Each chain is diagonalised once. Every temperature then reuses the same eigenvectors with different Boltzmann weights.

In [ ]:
struct Chain
  W::Matrix{Float64}     # eigenvectors of the chain Hamiltonian (columns)
  eps::Vector{Float64}   # its eigenvalues
end

"Boltzmann weights e^{-beta eps} / Z: the state's populations in the chain's eigenbasis."
function boltzmann(ch::Chain, beta)
  w = exp.(-beta .* (ch.eps .- minimum(ch.eps)))
  return w ./ sum(w)
end

### The dataset and the loader
The loader reads the couplings J, rebuilds each chain Hamiltonian and diagonalises it. It then checks every state's energy against the file.

In [ ]:
struct Dataset
  n::Int
  chains::Vector{Chain}
  chain::Vector{Int}      # which chain each state comes from
  group::Vector{String}   # "XX:3" = model XX, draw 3 (one per chain)
  model::Vector{String}
  kT::Vector{Float64}
  beta::Vector{Float64}
  y::Vector{Float64}      # -1 = XX, +1 = XXX
end

function chain_hamiltonian(n, J, xxx::Bool)
  H = zeros(2^n, 2^n)
  for i in 1:(n - 1)
    H .+= J[i] .* (pauli(n, [(i, 'X'), (i + 1, 'X')]) .+ pauli(n, [(i, 'Y'), (i + 1, 'Y')]))
    xxx && (H .+= J[i] .* pauli(n, [(i, 'Z'), (i + 1, 'Z')]))
  end
  return H
end


"Read the dataset and diagonalise every chain (~1 min)."
function load_dataset(path)
  rows = []
  n = 0
  h5open(path, "r") do f
    n = Int(read(attributes(f["meta"])["n"]))
    for name in sort(keys(f["samples"]))
      g = f["samples"][name]
      a = attributes(g)
      push!(rows, (model=String(read(a["model"])), draw=Int(read(a["draw"])),
                   kT=Float64(read(a["kT"])), beta=Float64(read(a["beta"])),
                   J=Vector{Float64}(read(g["J"])), energy=Float64(read(a["energy_ed"]))))
    end
  end

  group = ["$(r.model):$(r.draw)" for r in rows]
  groups = unique(group)
  chains = map(groups) do g
    r = rows[findfirst(==(g), group)]
    F = eigen(Symmetric(chain_hamiltonian(n, r.J, r.model == "XXX")))
    Chain(F.vectors, F.values)
  end
  chain = [findfirst(==(g), groups) for g in group]

  # sanity check: our rebuilt states have the energies the dataset generator recorded
  dE = maximum(abs(dot(boltzmann(chains[chain[m]], r.beta), chains[chain[m]].eps) - r.energy)
               for (m, r) in enumerate(rows))
  dE < 1e-8 || error("rebuilt states disagree with the file (max |dE| = $dE)")

  model = [r.model for r in rows]
  return Dataset(n, chains, chain, group, model, [r.kT for r in rows], [r.beta for r in rows],
                 [m == "XXX" ? 1.0 : -1.0 for m in model])
end

### Train / test split by state
The 840 states are split at random, state by state: 80 XX and 80 XXX states (160 in total) go to the test set and the other 680 to training. Temperatures and chains are mixed freely, so a test state usually shares its chain (its couplings J, at a different temperature) with some training states. The earlier version in `archive/` held out whole chains instead.

In [ ]:
"""
Random split over all 840 states, stratified by class: `test_per_class` XX states
and `test_per_class` XXX states go to the test set, the rest to training.
Chains and temperatures are not kept together.
"""
function split_by_state(ds::Dataset; test_per_class=80, seed=20260923)
  rng = MersenneTwister(seed)
  test = falses(length(ds.y))
  for cls in ("XX", "XXX")
    test[shuffle(rng, findall(ds.model .== cls))[1:test_per_class]] .= true
  end
  return findall(.!test), findall(test)
end

## 4. Training

### Settings
These are the settings folder 2 chose by cross-validation for Algorithm 8 (with its split by chain); they are kept unchanged here.

In [ ]:
using Optimisers, Printf

const DATA = joinpath(HERE, "..", "data", "xx_xxx_thermal_states", "xx_xxx_n10.h5")
const RESULTS = mkpath(joinpath(HERE, "results"))

const T = 1.0                                # activation width of GReLU_T
const L2 = 1e-4                              # weight decay
const STEPS, LR, SEED = 200, 0.05, 20260923  # Adam steps, learning rate, seed
const CHECKPOINTS = [0, 10, 20, 50, 100, 200]          # steps at which the circuit is simulated
const VALUE_RUNS = [10^2, 10^3, 10^4, 10^5, 10^6]     # value-block runs (cheap)
const GRAD_RUNS = [10, 30, 100, 300]                  # full Algorithm 8 runs (~30 ms each)

writecsv(path, header, rows) = open(io -> (println(io, join(header, ","));
                                           foreach(r -> println(io, join(r, ",")), rows)), path, "w")

### Load and split
This step takes about a minute, mostly diagonalising 84 dense 1024 × 1024 Hamiltonians.

In [ ]:
println("loading data ...")
ds = load_dataset(DATA)
train, test = split_by_state(ds)
@printf("  %d states from %d chains;  train %d states (from %d chains),  test %d states (from %d chains)\n",
        length(ds.y), length(ds.chains), length(train), length(unique(ds.chain[train])),
        length(test), length(unique(ds.chain[test])))

accuracy(score, idx) = mean((score .>= 0) .== (ds.y[idx] .> 0))
term_names, terms = neuron_terms(ds.n)
println("  neuron: ", length(terms), " terms: ", join(term_names, " "))

### Train the neuron and record θ
Adam runs on loss + (λ/2)|θ|² with the exact gradient. **θ and the exact loss gradient are stored at every step.** Training loss and test accuracy are printed every 10 steps. This takes about 5 minutes.

In [ ]:
"""
Adam on loss + L2/2 |theta|^2 with the exact gradient.  Returns the final
weights, theta and the loss gradient at every step (column s+1 = step s), and
a history of loss and test accuracy every 10 steps.
"""
function train_neuron(terms)
  theta = 0.1 .* randn(MersenneTwister(SEED), length(terms))
  opt = Optimisers.setup(Optimisers.Adam(LR), theta)
  thetas, grads = zeros(length(terms), STEPS + 1), zeros(length(terms), STEPS + 1)
  history = []
  for step in 0:STEPS
    L, g = square_loss_grad(ds, train, terms, theta, T)
    thetas[:, step + 1], grads[:, step + 1] = theta, g
    if step % 10 == 0
      acc = accuracy(scores(ds, test, terms, theta, T), test)
      push!(history, (step, L, acc))
      @printf("  step %3d   train loss %.4f   test accuracy %.3f   ||theta||_1 %.2f\n",
              step, L, acc, norm(theta, 1))
    end
    step == STEPS && break
    opt, theta = Optimisers.update!(opt, theta, g .+ L2 .* theta)
  end
  return (; theta=copy(theta), thetas, grads, history)
end

model = train_neuron(terms)

writecsv(joinpath(RESULTS, "history.csv"), ["step", "train_loss", "test_accuracy"], model.history)
writecsv(joinpath(RESULTS, "weights.csv"), ["term", "theta"], zip(term_names, model.theta))
writecsv(joinpath(RESULTS, "theta_trajectory.csv"), ["step", "term", "theta", "grad"],
         [(s, term_names[j], model.thetas[j, s + 1], model.grads[j, s + 1]) for s in 0:STEPS for j in eachindex(term_names)])

## 5. Circuit check: the simulated circuit against the exact values

At each checkpoint the θ recorded during training is fed into the simulated circuit. There are two checks:

1. **Value block alone.** It needs only the populations ⟨E|ρ_m|E⟩, so it can be run up to 10⁶ times on one training state. Its running average should converge to the exact output Tr[GReLU(H)ρ_m] as 1/√N.
2. **Full Algorithm 8 gradient.** Each run costs a few dense 1024 × 1024 matrix products (~30 ms), so only 300 runs are made per checkpoint. From them we measure the spread of one run's output and convert it into the **number of runs needed for a 10% relative error** in the gradient.

The run outputs are scaled by ‖θ‖₁ and ‖θ‖₁² (Section 2). Training grows ‖θ‖₁ and shrinks the gradient, so the number of runs needed grows very fast. This is why training here uses the exact gradient, and why folder 2 found that training on circuit shots fails. This section takes about 2 minutes.

In [ ]:
C = pauli_expectations(ds, train, terms)
i0 = findfirst(i -> ds.y[train[i]] > 0, eachindex(train))     # one XXX training state for the value check
@printf("value check on training state %s at kT = %.3g\n", ds.group[train[i0]], ds.kT[train[i0]])
rng = MersenneTwister(SEED + 2)
vrows, grows, nrows = [], [], []
for step in CHECKPOINTS
  th, g = model.thetas[:, step + 1], model.grads[:, step + 1]
  nt1 = norm(th, 1)
  # 1. the value block alone
  est, exact = value_block_estimates(ds, train[i0], C[i0, :], terms, th, T; runs=VALUE_RUNS, rng)
  append!(vrows, [(step, nt1, N, v, exact) for (N, v) in zip(VALUE_RUNS, est)])
  # 2. the full Algorithm 8 gradient
  out = alg8_circuit_runs(ds, train, C, terms, th, T; nruns=maximum(GRAD_RUNS), rng)
  append!(grows, [(step, nt1, N, norm(vec(mean(out[:, 1:N]; dims=2)) - g) / norm(g)) for N in GRAD_RUNS])
  noise = sqrt(sum(var(out; dims=2)))          # spread of one run's output (norm over j)
  n10 = (noise / (0.1 * norm(g)))^2            # runs needed for a 10% relative error
  push!(nrows, (step, nt1, norm(g), noise, n10))
  @printf("  step %3d  ||theta||_1 %5.2f   value: exact %.3f, after 10^6 runs %.3f   gradient: |g| %.1e, one-run spread %.1e  ->  %.1e runs for 10%% error\n",
          step, nt1, exact, est[end], norm(g), noise, n10)
end
writecsv(joinpath(RESULTS, "circuit_value.csv"), ["step", "theta_l1", "runs", "estimate", "exact"], vrows)
writecsv(joinpath(RESULTS, "circuit_gradient.csv"), ["step", "theta_l1", "runs", "rel_error"], grows)
writecsv(joinpath(RESULTS, "circuit_cost.csv"), ["step", "theta_l1", "grad_norm", "run_spread", "runs_for_10pct"], nrows)

## 6. Testing

### Exact outputs
The decision score Tr[GReLU(H)ρ] − ½ is computed for every test state. Exact outputs correspond to infinitely many copies of each state.

In [ ]:
sc = scores(ds, test, terms, model.theta, T)
@printf("test accuracy %.3f   ||theta||_1 = %.1f   bias theta_I = %.2f\n",
        accuracy(sc, test), norm(model.theta, 1), model.theta[1])
writecsv(joinpath(RESULTS, "test_scores.csv"), ["chain", "kT", "y", "score"],
         [(ds.group[i], ds.kT[i], Int(ds.y[i]), sc[j]) for (j, i) in enumerate(test)])

### Classify by firing
Accuracy is measured against the number of copies of each test state, averaged over 10 repeats.

In [ ]:
const COPIES = [2, 8, 32, 128, 512, 2048]
const REPS = 10
rng = MersenneTwister(SEED + 1)
E, V = eigen(Symmetric(hamiltonian(terms, model.theta)))
pops = populations(ds, test, V)
firing = map(COPIES) do N
  mean(mean(classify_by_firing(E, pops[:, j], T, N, rng) == ds.y[i]
            for (j, i) in enumerate(test)) for _ in 1:REPS)
end
println(join((@sprintf("%d copies: %.3f", N, a) for (N, a) in zip(COPIES, firing)), "   "))
writecsv(joinpath(RESULTS, "firing.csv"), ["copies", "accuracy"], zip(COPIES, firing))

## 7. Figures

The figures are drawn from the CSVs just written to `results/`, so this section can be re-run on its own after the path cell at the top. Each figure is shown inline and saved to `figures/`.

In [ ]:
using Plots, Statistics

const RES = joinpath(HERE, "results")
const FIG = mkpath(joinpath(HERE, "figures"))

"Read a CSV written above into named columns (numbers parsed where possible)."
function readcsv(name)
  lines = readlines(joinpath(RES, name))
  head = Symbol.(split(lines[1], ","))
  rows = [split(l, ",") for l in lines[2:end]]
  col(i) = (v = getindex.(rows, i); all(x -> tryparse(Float64, x) !== nothing, v) ? parse.(Float64, v) : String.(v))
  return NamedTuple{Tuple(head)}(Tuple(col(i) for i in eachindex(head)))
end

termkind(n) = n == "I" ? "I" : count(isletter, n) == 1 ? "Z" : string(n[1], n[1])   # "X3X4" -> "XX"
const KINDS = ["I", "Z", "ZZ", "XX", "YY"]
const KCOLOR = Dict("I" => :black, "Z" => :seagreen, "ZZ" => :crimson, "XX" => :royalblue, "YY" => :darkorange)
default(framestyle=:box, grid=false, dpi=200)

### Training loss and test accuracy

In [ ]:
h = readcsv("history.csv")
p1 = plot(h.step, h.train_loss, lw=2, c=:darkorange, yscale=:log10, xlabel="Adam step",
          ylabel="training loss", title="squared loss", legend=false)
p2 = plot(h.step, h.test_accuracy, lw=2, c=:darkorange, ylim=(0.4, 1.03), xlabel="Adam step",
          ylabel="test accuracy", title="test accuracy (exact outputs)", legend=false)
fig = plot(p1, p2, layout=(1, 2), size=(900, 350), margin=5Plots.mm)
savefig(fig, joinpath(FIG, "1_training.png"))
fig

### How θ moves during training
One panel per kind of term. Each line is one θⱼ, coloured by its site from dark (site 1) to light. The last panel shows ‖θ‖₁, which sets the noise of the circuit (Section 2).

In [ ]:
tr = readcsv("theta_trajectory.csv")
steps = sort(unique(tr.step))
traj(n) = tr.theta[tr.term .== n]                 # rows are written in step order
termnames = tr.term[tr.step .== 0]
ps = map(KINDS) do k
  ns = filter(n -> termkind(n) == k, termnames)
  q = plot(xlabel="Adam step", ylabel="θ", title=k == "I" ? "bias (identity)" : "$k terms", legend=false)
  hline!(q, [0], c=:gray70, lw=0.5)
  for (i, n) in enumerate(ns)
    plot!(q, steps, traj(n), lw=1.5, c=k == "I" ? :black : get(cgrad(:viridis), (i - 1) / max(length(ns) - 1, 1) * 0.85))
  end
  q
end
l1 = [sum(abs, tr.theta[tr.step .== s]) for s in steps]
push!(ps, plot(steps, l1, lw=2, c=:black, xlabel="Adam step", ylabel="‖θ‖₁", title="‖θ‖₁", legend=false))
fig = plot(ps..., layout=(2, 3), size=(1050, 600), margin=4Plots.mm)
savefig(fig, joinpath(FIG, "2_theta_trajectories.png"))
fig

### θ and its gradient as heatmaps
Left: θⱼ at every step (rows = terms, grouped by kind). Right: the exact loss gradient ∂L/∂θⱼ that drives each Adam update (symmetric log colour scale).

In [ ]:
Th = reduce(hcat, [tr.theta[tr.step .== s] for s in steps])
G = reduce(hcat, [tr.grad[tr.step .== s] for s in steps])
slog(x) = sign(x) * log10(1 + abs(x) / 1e-4)       # symmetric log, linear below 1e-4
yt = (1:length(termnames), termnames)
c1 = maximum(abs, Th)
p1 = heatmap(steps, 1:length(termnames), Th, c=:RdBu, clims=(-c1, c1), yflip=true, yticks=yt,
             ytickfontsize=5, xlabel="Adam step", title="θⱼ")
c2 = maximum(abs, slog.(G))
p2 = heatmap(steps, 1:length(termnames), slog.(G), c=:PuOr, clims=(-c2, c2), yflip=true, yticks=yt,
             ytickfontsize=5, xlabel="Adam step", title="∂L/∂θⱼ  (sign · log₁₀(1 + |g|/10⁻⁴))")
fig = plot(p1, p2, layout=(1, 2), size=(1050, 520), margin=4Plots.mm)
savefig(fig, joinpath(FIG, "3_theta_heatmap.png"))
fig

### Final weights
Every term's trained weight, grouped by kind.

In [ ]:
w = readcsv("weights.csv")
lim = 1.1 * maximum(abs, w.theta)
p = plot(ylabel="θ", title="trained weights", size=(900, 400), legend=:bottomright, ylim=(-lim, lim),
         left_margin=5Plots.mm,
         xticks=(1:length(w.term), w.term), xrotation=90, xtickfontsize=6, bottom_margin=6Plots.mm)
for k in KINDS
  sel = termkind.(w.term) .== k
  bar!(p, findall(sel), w.theta[sel], bar_width=0.8, c=KCOLOR[k], lc=KCOLOR[k], label=k)
end
hline!(p, [0], c=:black, lw=0.5, label="")
savefig(p, joinpath(FIG, "4_weights.png"))
p

### The simulated circuit against the exact values
Left: the value block's relative error against the number of runs, one line per checkpoint (dark = early, light = late), with dotted 1/√N guides. Middle: the same for the full Algorithm 8 gradient (300 runs per checkpoint). Right: the number of circuit runs needed for a 10% error in the gradient, against ‖θ‖₁ along training.

In [ ]:
cv, cg, cc = readcsv("circuit_value.csv"), readcsv("circuit_gradient.csv"), readcsv("circuit_cost.csv")
cps = sort(unique(cv.step))
ccol(step) = get(cgrad(:viridis), 0.85 * (findfirst(==(step), cps) - 1) / max(length(cps) - 1, 1))
lab(step) = "step $(Int(step))"
p1 = plot(xscale=:log10, yscale=:log10, xlabel="value-block runs N", ylabel="|estimate − exact| / exact",
          title="value block, one state", legend=:bottomleft, legendfontsize=7)
p2 = plot(xscale=:log10, yscale=:log10, xlabel="Algorithm 8 runs N", ylabel="‖ĝ − g‖ / ‖g‖",
          title="full gradient", legend=false)
for step in cps
  rv, rg = cv.step .== step, cg.step .== step
  ev = abs.(cv.estimate[rv] .- cv.exact[rv]) ./ abs.(cv.exact[rv])
  plot!(p1, cv.runs[rv], ev, lw=2, marker=:circle, ms=3, c=ccol(step), label=lab(step))
  plot!(p1, cv.runs[rv], ev[1] .* sqrt.(cv.runs[rv][1] ./ cv.runs[rv]), ls=:dot, c=ccol(step), label="")
  plot!(p2, cg.runs[rg], cg.rel_error[rg], lw=2, marker=:circle, ms=3, c=ccol(step))
  plot!(p2, cg.runs[rg], cg.rel_error[rg][1] .* sqrt.(cg.runs[rg][1] ./ cg.runs[rg]), ls=:dot, c=ccol(step))
end
hline!(p2, [1], c=:gray, ls=:dash)
p3 = plot(cc.theta_l1, cc.runs_for_10pct, yscale=:log10, lw=1, c=:gray, marker=:circle, ms=5,
          mc=[ccol(st) for st in cc.step], msw=0, xlabel="‖θ‖₁", ylabel="runs for 10% gradient error",
          title="cost of the circuit gradient", legend=false,
          ylim=(minimum(cc.runs_for_10pct) / 10, maximum(cc.runs_for_10pct) * 100))
annotate!(p3, [(x, y * 2.5, text(lab(st), 7)) for (x, y, st) in zip(cc.theta_l1, cc.runs_for_10pct, cc.step)])
fig = plot(p1, p2, p3, layout=(1, 3), size=(1250, 410), margin=5Plots.mm, bottom_margin=8Plots.mm)
savefig(fig, joinpath(FIG, "5_circuit_check.png"))
fig

### Decision score of every test state against temperature

In [ ]:
t = readcsv("test_scores.csv")
p = plot(xscale=:log10, xlabel="kT", ylabel="Tr[GReLU(H) ρ] − ½", size=(600, 380),
         title="test states (red XXX, grey XX)", legend=:right)
for (y, c, lab) in ((-1, :gray40, "XX"), (1, :crimson, "XXX"))
  r = t.y .== y
  scatter!(p, t.kT[r] .* exp.(0.04 .* randn(count(r))), t.score[r], ms=3, msw=0, alpha=0.7, c=c, label=lab)
end
hline!(p, [0], ls=:dash, c=:black, label="")
savefig(p, joinpath(FIG, "6_test_scores.png"))
p

### Classifying by firing: accuracy against copies

In [ ]:
f = readcsv("firing.csv")
p = plot(f.copies, f.accuracy, lw=2, marker=:circle, c=:darkorange, xscale=:log2, label="fired neuron",
         xlabel="copies of each test state", ylabel="test accuracy", ylim=(0.45, 1.03),
         title="classifying by firing the neuron (Algorithm 5)", size=(600, 380), legend=:bottomright)
hline!(p, [0.5], ls=:dash, c=:gray, label="chance")
savefig(p, joinpath(FIG, "7_firing.png"))
p

## 8. Results and conclusions

Numbers from the run of 7 Oct 2026 (seed 20260923). Re-running the notebook should reproduce them exactly.

- **Exact test accuracy is 100% from step 10 onwards.** The final training loss is 0.017, ‖θ‖₁ = 34.5 and the bias θ_I = −0.25.
- **The split is by state, so this is an easy test.** The 160 test states come from 75 of the 84 chains, and every one of those chains also appears in training at other temperatures. The earlier version, which held out whole chains (`archive/`), also reached 100%. The task is easy because XXX chains are rotation-symmetric (⟨ZZ⟩ = ⟨XX⟩ = ⟨YY⟩), while in XX chains ⟨ZZ⟩ is much weaker.
- **What the neuron learns.** The ZZ weights are negative (−1.2 to −1.7) and the XX and YY weights positive (0.6 to 1.8, largest mid-chain), so the neuron compares ⟨XX⟩ + ⟨YY⟩ with ⟨ZZ⟩. The single-site Z weights are no use for this task: they swing to about ±0.4 in the first 20 steps, then decay to 0 by step ~100.
- **θ keeps moving after accuracy saturates.** ‖θ‖₁ rises from 3.7 to 34.5, and the loss is still falling slowly at step 200. The decision margin is smallest for XX states at the highest temperature, kT = 2 (score ≈ −0.1 to −0.3, against about −0.35 elsewhere).
- **The simulated circuit is unbiased but very noisy.** At step 0, 10⁶ runs of the value block give 0.584 against the exact 0.581; at later checkpoints the error after 10⁶ runs is 3–20%. Errors fall as 1/√N throughout. For the full gradient, a single run's spread is ~80 against ‖g‖ = 0.17 at step 0, so a 10% gradient error needs ~2 × 10⁷ runs. By step 200 this grows to ~10¹⁷, because the outputs scale with ‖θ‖₁ while the gradient shrinks. Training directly on circuit runs is only conceivable for the first few steps, which agrees with folder 2.
- **Classifying by firing** (Algorithm 5) gives 60% accuracy from 2 copies of each test state, 71% from 8, 87% from 32, 98% from 128 and 100% from 512.